# LLM-Speed Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LessUp/llm-speed/blob/main/notebooks/llm-speed-demo.ipynb)

This notebook demonstrates the key features of **LLM-Speed**, a CUDA kernel library for LLM inference:

- FlashAttention: O(N) memory complexity attention
- Tensor Core GEMM: Hardware-accelerated matrix multiplication
- Performance comparison with PyTorch reference

## 1. Setup

First, let's check the GPU environment and install dependencies.

In [ ]:
# Check GPU
!nvidia-smi --query-gpu=name,compute_cap --format=csv

In [ ]:
# Install PyTorch (if not already installed)
try:
    import torch
    print(f"PyTorch {torch.__version__} already installed")
except ImportError:
    !pip install torch --index-url https://download.pytorch.org/whl/cu121
    import torch

In [ ]:
# Clone and install LLM-Speed
!git clone https://github.com/LessUp/llm-speed.git
%cd llm-speed
!pip install -e .

## 2. FlashAttention Demo

FlashAttention achieves **O(N) memory complexity** instead of O(N²) for standard attention.

In [ ]:
import torch
from cuda_llm_ops import flash_attention, naive_attention

# Configuration
batch, heads = 2, 8
seq_len, head_dim = 2048, 64

print(f"Configuration: batch={batch}, heads={heads}, seq_len={seq_len}, head_dim={head_dim}")
print(f"Attention matrix size: {seq_len * seq_len * 4 / 1024 / 1024:.1f} MB (if stored)")

# Create inputs
q = torch.randn(batch, heads, seq_len, head_dim, device='cuda', dtype=torch.float16)
k = torch.randn_like(q)
v = torch.randn_like(q)

# FlashAttention - O(N) memory!
output_flash = flash_attention(q, k, v, is_causal=True)
print(f"\nFlashAttention output shape: {output_flash.shape}")

### Memory Comparison

Let's compare memory usage between naive and FlashAttention:

In [ ]:
def measure_memory(func, *args, **kwargs):
    """Measure peak memory usage of a function."""
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()
    
    result = func(*args, **kwargs)
    torch.cuda.synchronize()
    
    peak_mb = torch.cuda.max_memory_allocated() / 1024 / 1024
    return result, peak_mb

# Measure FlashAttention
_, mem_flash = measure_memory(flash_attention, q, k, v, is_causal=True)

# Estimate naive attention memory (O(N²))
mem_naive_estimate = batch * heads * seq_len * seq_len * 2 / 1024 / 1024  # FP16

print(f"Memory Usage:")
print(f"  FlashAttention: {mem_flash:.1f} MB (actual)")
print(f"  Naive (estimated): {mem_naive_estimate:.1f} MB")
print(f"  Savings: {mem_naive_estimate / mem_flash:.1f}x")

### Correctness Verification

Compare FlashAttention output with PyTorch reference:

In [ ]:
# PyTorch reference
output_torch = torch.nn.functional.scaled_dot_product_attention(q, k, v, is_causal=True)

# Compare
max_diff = (output_flash - output_torch).abs().max().item()
mean_diff = (output_flash - output_torch).abs().mean().item()

print(f"Correctness Check:")
print(f"  Max absolute difference: {max_diff:.6f}")
print(f"  Mean absolute difference: {mean_diff:.6f}")
print(f"  Status: {'✅ PASS' if max_diff < 1e-3 else '❌ FAIL'}")

## 3. Tensor Core GEMM Demo

Hardware-accelerated matrix multiplication using Tensor Cores.

In [ ]:
from cuda_llm_ops import tensor_core_gemm

# Matrix dimensions
M, N, K = 2048, 2048, 2048

print(f"Matrix dimensions: M={M}, N={N}, K={K}")
print(f"FLOPs: {2 * M * N * K / 1e9:.1f} GFLOPs")

# Create FP16 input matrices
a = torch.randn(M, K, device='cuda', dtype=torch.float16)
b = torch.randn(K, N, device='cuda', dtype=torch.float16)

# Tensor Core GEMM (FP16 input → FP32 output)
c = tensor_core_gemm(a, b)
print(f"\nOutput shape: {c.shape}")
print(f"Output dtype: {c.dtype} (FP32 accumulation)")

### Performance Benchmark

In [ ]:
import time

def benchmark_gemm(func, a, b, iterations=100, warmup=10):
    """Benchmark GEMM operation."""
    # Warmup
    for _ in range(warmup):
        _ = func(a, b)
    torch.cuda.synchronize()
    
    # Benchmark
    start = time.perf_counter()
    for _ in range(iterations):
        _ = func(a, b)
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    
    avg_ms = elapsed / iterations * 1000
    flops = 2 * a.shape[0] * b.shape[1] * a.shape[1]
    tflops = flops / (avg_ms * 1e-3) / 1e12
    
    return avg_ms, tflops

# Benchmark Tensor Core GEMM
ms_tc, tflops_tc = benchmark_gemm(tensor_core_gemm, a, b)

# Benchmark PyTorch matmul
ms_torch, tflops_torch = benchmark_gemm(torch.matmul, a, b)

print(f"Performance Comparison:")
print(f"  Tensor Core GEMM: {ms_tc:.3f} ms ({tflops_tc:.1f} TFLOPS)")
print(f"  PyTorch matmul:   {ms_torch:.3f} ms ({tflops_torch:.1f} TFLOPS)")
print(f"  Efficiency:       {tflops_tc / tflops_torch * 100:.1f}% of PyTorch")

## 4. Scaling Analysis

How does FlashAttention scale with sequence length?

In [ ]:
import matplotlib.pyplot as plt

seq_lengths = [512, 1024, 2048, 4096, 8192]
times_flash = []
times_torch = []

for seq_len in seq_lengths:
    q = torch.randn(2, 8, seq_len, 64, device='cuda', dtype=torch.float16)
    k = torch.randn_like(q)
    v = torch.randn_like(q)
    
    # Benchmark FlashAttention
    _, ms_flash = benchmark_gemm(lambda q, k, v: flash_attention(q, k, v), q, k, v)
    times_flash.append(ms_flash)
    
    # Benchmark PyTorch
    _, ms_torch = benchmark_gemm(
        lambda q, k, v: torch.nn.functional.scaled_dot_product_attention(q, k, v),
        q, k, v
    )
    times_torch.append(ms_torch)
    
    print(f"Seq {seq_len}: Flash={ms_flash:.2f}ms, Torch={ms_torch:.2f}ms, Speedup={ms_torch/ms_flash:.2f}x")

# Plot
plt.figure(figsize=(10, 6))
plt.plot(seq_lengths, times_torch, 'o-', label='PyTorch SDPA', linewidth=2)
plt.plot(seq_lengths, times_flash, 's-', label='FlashAttention', linewidth=2)
plt.xlabel('Sequence Length')
plt.ylabel('Time (ms)')
plt.title('FlashAttention vs PyTorch SDPA')
plt.legend()
plt.grid(True)
plt.show()

## 5. Summary

Key takeaways:

- **FlashAttention**: O(N) memory complexity enables processing very long sequences
- **Tensor Core GEMM**: Hardware acceleration for matrix operations
- **Correctness**: Output matches PyTorch reference within FP16 precision
- **Performance**: Competitive with or faster than PyTorch for attention operations